In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Cargamos el modelo
from models.actions.cpmp_transformer_V1 import CPMPTransformer
from training.training import load_model

model_name = "best_sl"
model = load_model(CPMPTransformer, model_name)

# Configuramos el adaptador para la entrada del modelo
from generation.adapters.input import EnrichedLayoutAdapter, Layout4DAdapterV1
from generation.adapters.input.stack_features import StackFeaturesAdapterV1

input_adapter = EnrichedLayoutAdapter(Layout4DAdapterV1, StackFeaturesAdapterV1)

# Solvers

## Modelo (Greedy)

In [5]:
from solvers.model import ModelSolver

instance_file = "benchmarks/5-5/data5-5-1.dat"
max_steps = 100
H = 7

solver = ModelSolver(model, input_adapter)
eval = solver.solve(instance_file, H, max_steps)

print(eval)

(True, 30, 0.08557296599974507)


In [ ]:
from solvers.model import ModelSolver
from solvers.utils import summary

folder = "benchmarks/10-10"
max_steps = 500
H = 12

solver = ModelSolver(model, input_adapter)
results = solver.solve_from_folder(folder, H, max_steps)
summary(results)

Instancias resueltas: 39/40
Promedio de pasos: 182.10


## BSG normal

In [ ]:
from solvers.bsg import BSGModelSolver
from solvers.utils import summary

folder = "benchmarks/10-10"
max_steps = 500
H = 12
w = 2

solver = BSGModelSolver(model, input_adapter, w)
results = solver.solve_from_folder(folder, H, max_steps)
summary(results)

KeyboardInterrupt: 

## BSG + Modelo predictor de costo

In [ ]:
from solvers.bsg import BSGCostPredictorSolver
from solvers.utils import summary
from models.cost.cost_predictor import CostPredictorTransformer
from training.training import load_model

## Cargar modelo predictor del costo
model_name = "cost"
cost_model = load_model(CostPredictorTransformer, model_name)

## Cargar modelo predictor de acciones
action_model = model

folder = "benchmarks/5-5"
max_steps = 50
H = 7
w = 2

solver = BSGCostPredictorSolver(action_model, cost_model, EnrichedLayoutAdapter(Layout4D2FAdapter, StackFeatures3FAdapter), w)
results = solver.solve_from_folder(folder, H, max_steps)
summary(results)

TypeError: CostPredictorTransformer.forward() takes 3 positional arguments but 5 were given

## FRG

In [ ]:
from solvers.FRG import FRGSolver
from solvers.utils import summary

folder = "benchmarks"
max_steps = 50
H = 7

solver = FRGSolver()
results = solver.solve_from_folder(folder, H, max_steps)
summary(results)

Instancias resueltas: 40/40
Promedio de pasos: 35.10


## BS-FRG

In [ ]:
from solvers.bsg.bsg_frg import BSGFRGSolver
from solvers.utils import summary

folder = "benchmarks"
max_steps = 1000
H = 7

solver = BSGFRGSolver(100)
results = solver.solve_from_folder(folder, H, max_steps)
summary(results)

Instancias resueltas: 40/40
Promedio de pasos: 27.23


# BSG Híbrido (Modelo + FRG)

In [ ]:
from solvers.bsg.bsg_hybrid import BSGHybridSolver
from solvers.utils import summary

folder = "benchmarks"
max_steps = 50
H = 7

solver = BSGHybridSolver(model, input_adapter, w=2)
results = solver.solve_from_folder(folder, H, max_steps)
summary(results)

Instancias resueltas: 40/40
Promedio de pasos: 28.55


# Doble esfuerzo

In [6]:
from solvers.double_effort import DoubleEffortSolver
from solvers.bsg import BSGModelSolver
from solvers.utils import summary

folder = "benchmarks"
max_steps = 50
H = 7
max_time = 1

base_solver = BSGModelSolver(model, input_adapter, w=1)
dse_solver = DoubleEffortSolver(base_solver, max_time)
results = dse_solver.solve_from_folder(folder, H, max_steps)
summary(results)

Instancias resueltas: 39/40
Promedio de pasos: 26.28


# Evaluadores

In [7]:
# Cargamos el modelo
from models.cpmp_transformer_v6 import CPMPTransformer
from training.training import load_model

model_name = "best_rl"
model = load_model(CPMPTransformer, model_name)

# Configuramos el adaptador para la entrada del modelo
from generation.adapters.input import EnrichedLayoutAdapter, Layout4D3FAdapter
from generation.adapters.input.stack_features import StackFeatures3FAdapter

input_adapter = EnrichedLayoutAdapter(Layout4D3FAdapter, StackFeatures3FAdapter)

## Beam search

In [ ]:
from solvers import BSGFRGSolver, BSGModelSolver, BSGHybridSolver, BSGCostPredictorSolver
from evaluators.evaluator import solver_eval

# Cargamos los modelos
from models.cpmp_transformer_v6 import CPMPTransformer
from models.cost.cost_predictor import CostPredictorTransformer
from training.training import load_model
action_model_name = "best_rl"
action_model = load_model(CPMPTransformer, model_name)
cost_model_name = "cost"
cost_model = load_model(CostPredictorTransformer, cost_model_name)

# Configuramos el adaptador para la entrada del modelo
from generation.adapters.input import EnrichedLayoutAdapter, Layout4D3FAdapter
from generation.adapters.input.stack_features import StackFeatures3FAdapter
input_adapter = EnrichedLayoutAdapter(Layout4D3FAdapter, StackFeatures3FAdapter)

# Configuramos las instancias y solvers a evaluar
folder = "benchmarks"
H = 7
max_steps = 50
w = 32

solvers = [
    BSGFRGSolver(w),
    BSGModelSolver(model, input_adapter, w),
    BSGHybridSolver(model, input_adapter, w),
    BSGCostPredictorSolver(action_model, cost_model, input_adapter, w)
]

solver_eval(solvers, folder, H, max_steps)

Solver                    | Instancia  | Avg Solved   | Avg Steps    | Avg Time    
-----------------------------------------------------------------------------------
BSGFRGSolver              | 40         | 1.0000       | 28.3250      | 0.0349      
BSGModelSolver            | 40         | 1.0000       | 25.4250      | 22.2249     
BSGHybridSolver           | 40         | 1.0000       | 27.2000      | 10.7820     
BSGCostPredictorSolver    | 40         | 1.0000       | 25.9750      | 1.8551      


,instance,Solved BSGFRGSolver,Steps BSGFRGSolver,Time BSGFRGSolver,Solved BSGModelSolver,Steps BSGModelSolver,Time BSGModelSolver,Solved BSGHybridSolver,Steps BSGHybridSolver,Time BSGHybridSolver,Solved BSGCostPredictorSolver,Steps BSGCostPredictorSolver,Time BSGCostPredictorSolver
0,data5-5-28.dat,True,32,0.060664,True,26,27.917397,True,30,11.928275,True,30,2.252965
1,data5-5-7.dat,True,27,0.025130,True,25,18.201157,True,27,11.078971,True,25,1.770638
2,data5-5-2.dat,True,32,0.047394,True,29,28.758024,True,33,12.755302,True,30,1.929227
3,data5-5-37.dat,True,26,0.038438,True,22,16.576647,True,24,8.672888,True,23,1.687172
4,data5-5-3.dat,True,31,0.036706,True,25,20.132228,True,25,9.223148,True,27,1.823292
5,data5-5-32.dat,True,28,0.031039,True,25,19.919973,True,27,10.371107,True,26,1.762595
6,data5-5-16.dat,True,26,0.020703,True,24,18.831453,True,25,10.134168,True,25,1.786592
7,data5-5-18.dat,True,28,0.033882,True,24,18.315421,True,27,10.170035,True,26,1.829230
8,data5-5-38.dat,True,25,0.021847,True,23,16.324744,True,24,9.474299,True,23,1.815137
9,data5-5-19.dat,True,30,0.040898,True,26,24.623154,True,30,12.662644,True,26,1.983322


## Double Search Effort

In [ ]:
from solvers import BSGFRGSolver, BSGModelSolver, BSGHybridSolver, BSGCostPredictorSolver
from evaluators.dse_evaluator import dse_eval

# Cargamos los modelos
from models.cpmp_transformer_v6 import CPMPTransformer
from models.cost.cost_predictor import CostPredictorTransformer
from training.training import load_model
action_model_name = "best_rl"
action_model = load_model(CPMPTransformer, model_name)
cost_model_name = "cost"
cost_model = load_model(CostPredictorTransformer, cost_model_name)

# Configuramos el adaptador para la entrada del modelo
from generation.adapters.input import EnrichedLayoutAdapter, Layout4D3FAdapter
from generation.adapters.input.stack_features import StackFeatures3FAdapter
input_adapter = EnrichedLayoutAdapter(Layout4D3FAdapter, StackFeatures3FAdapter)

# Configuramos las instancias y solvers a evaluar
folder = "benchmarks"
H = 7
max_steps = 50
w = 1
max_time = 10

solvers = [
    BSGFRGSolver(w),
    BSGModelSolver(model, input_adapter, w),
    BSGHybridSolver(model, input_adapter, w),
    BSGCostPredictorSolver(action_model, cost_model, input_adapter, w)
]

dse_eval(solvers, folder, H, max_steps, max_time)

Solver                         | Instancia  | Avg Solved   | Avg Steps   
-------------------------------------------------------------------------
DSE BSGFRGSolver               | 40         | 1.0000       | 26.5750     


,instance,Solved DSE BSGFRGSolver,Steps DSE BSGFRGSolver
0,data5-5-28.dat,True,30
1,data5-5-7.dat,True,26
2,data5-5-2.dat,True,30
3,data5-5-37.dat,True,24
4,data5-5-3.dat,True,25
5,data5-5-32.dat,True,27
6,data5-5-16.dat,True,24
7,data5-5-18.dat,True,27
8,data5-5-38.dat,True,22
9,data5-5-19.dat,True,27
